# 🧑‍🦲 光头强专属 RVC 音色模型 · 训练笔记本

> 训练自己的光头强音色转换模型（RVC v2），之后可用「微软情绪 TTS + RVC 转换」
> 做出带**动漫配音感**的光头强语音。**免费 GPU + 无限次使用**。

## 流程总览

1. 切到 **T4 GPU**（代码执行程序 → 更改运行时类型）
2. 依次运行每个格子：装环境 → 下预训练模型 → 上传素材 → 人声分离 → 切段 → 提特征 → **训练（约 30-60 分钟）** → 提取索引
3. 训练完在同一个笔记本里**直接转换一段底音试听效果**
4. 下载训练好的模型（.pth + .index），以后配合微软 TTS 无限次使用

## 素材说明

配套的 `rvc_material.zip`（桌面「中转站」文件夹里，约 9 分钟干净光头强配音，
来自 B 站配音对比视频）会在第 3 格上传。


## 第 0 步：确认 GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
print(r)
assert "T4" in r or "Tesla" in r, "请先在 代码执行程序→更改运行时类型 里选择 T4 GPU！" 

## 第 1 步：安装 RVC WebUI 环境（约 3 分钟）

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd /content/rvc
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip install -q demucs 2>&1 | tail -2
print("环境安装完成")

## 第 2 步：下载预训练模型（约 200MB）

In [ ]:
%cd /content/rvc
import os
os.makedirs("assets/hubert_base", exist_ok=True)
os.makedirs("assets/rmvpe", exist_ok=True)
os.makedirs("assets/pretrained_v2", exist_ok=True)

!wget -q -O assets/hubert_base/hubert_base.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/hubert_base.pt
!wget -q -O assets/rmvpe/rmvpe.pt https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt
!wget -q -O assets/pretrained_v2/f0G40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0G40k.pth
!wget -q -O assets/pretrained_v2/f0D40k.pth https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/pretrained_v2/f0D40k.pth

!ls -lh assets/hubert_base assets/rmvpe assets/pretrained_v2
print("预训练模型下载完成")

## 第 3 步：下载素材并解压

素材已托管在 GitHub，自动下载（约 9 分钟光头强配音，无需手动上传）。

In [ ]:
%cd /content/rvc
!wget -q https://raw.githubusercontent.com/haiyulangman/gtq-voice/main/rvc_material.zip -O rvc_material.zip
!unzip -o -q rvc_material.zip -d raw_material
!ls -lh raw_material/

## 第 4 步：人声分离（demucs，T4 上约 5-10 分钟）

把素材里的背景音乐去掉，只留人声（训练质量的关键）。

In [ ]:
%cd /content/rvc
import glob, os, shutil

files_in = glob.glob("raw_material/*.m4a") + glob.glob("raw_material/*.mp3") + glob.glob("raw_material/*.wav")
print("待分离:", [os.path.basename(f) for f in files_in])

os.makedirs("raw_vocals", exist_ok=True)
for f in files_in:
    !demucs --two-stems=vocals -n htdemucs "{f}" -o separated 2>&1 | tail -1

# 收集分离出的人声
vocals = glob.glob("separated/htdemucs/**/vocals.wav", recursive=True)
print(f"分离出 {len(vocals)} 个人声文件")
for i, v in enumerate(vocals):
    shutil.copy(v, f"raw_vocals/vocals_{i:02d}.wav")
!ls -lh raw_vocals/

## 第 5 步：切段预处理（静音检测切 3.7 秒片段）

In [ ]:
%cd /content/rvc
!python train/preprocess.py raw_vocals 40000 2 logs/gtq False 3.7 2>&1 | tail -5
import os
n = len(os.listdir("logs/gtq/0_gt_wavs"))
print(f"切段完成：共 {n} 个训练片段")

## 第 6 步：提取特征（F0 rmvpe + HuBERT，约 5-10 分钟）

In [ ]:
%cd /content/rvc
# F0 提取（GPU 加速）
!python train/dataset/extract_f0.py cuda 1 0 0 logs/gtq True 2>&1 | tail -3
# HuBERT 特征提取
!python train/dataset/extract_hubert_feature.py cuda 1 0 0 logs/gtq True v2 2>&1 | tail -3
print("特征提取完成")
!ls logs/gtq/

## 第 7 步：训练模型（约 30-60 分钟，请勿关闭页面）

- v2 架构、40k 采样率、batch 8、共 300 轮、每 50 轮保存一次
- 训练中会打印每个 epoch 的 loss，不用管具体数字，跑完即可
- 若提示显存不足，把 `-bs 8` 改成 `-bs 4` 重跑本格

In [ ]:
%cd /content/rvc
!python train/train.py -e gtq -sr 40k -f0 1 -bs 8 -g 0 -te 300 -se 50 \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth \
  -l 1 -c 0 -sw 1 -v v2 2>&1 | tail -20
print("训练结束")
!ls -lh logs/gtq/weights/

## 第 8 步：提取音色索引（约 3-5 分钟，提升相似度）

In [ ]:
%cd /content/rvc
!python train/train_index.py gtq v2 logs 2 2>&1 | tail -5
print("索引提取完成")
!ls -lh logs/gtq/

## 第 9 步：试听效果（上传底音 → 转换 → 播放）

先随便上传一个**男声说话音频**（mp3/wav 均可，10 秒以上），
或者用微软 TTS 底音（见下一步说明）转换后试听光头强音色。

In [ ]:
%cd /content/rvc
from google.colab import files
import glob, os, shutil

print("请上传一个男声音频文件作为测试底音（mp3/wav）")
up = files.upload()
for name in up:
    if name.endswith((".mp3", ".wav", ".m4a")):
        shutil.copy(name, "test_input" + os.path.splitext(name)[1])

# 把训练好的最新模型复制到权重目录
weights = sorted(glob.glob("logs/gtq/weights/gtq_e*.pth"))
latest = [w for w in weights if "e" in os.path.basename(w)]
print("模型文件:", [os.path.basename(w) for w in latest])
import subprocess
best = latest[-1]
shutil.copy(best, "assets/weights/gtq.pth")

# 转换
test_in = glob.glob("test_input.*")[0]
!python infer/cli.py --model gtq --input "{test_in}" --output test_output \
    --f0-method rmvpe --index-rate 0.75 --format wav 2>&1 | tail -3

# 试听
import IPython.display as ipd
out = glob.glob("test_output/*.wav")[0]
print("转换完成：", out)
ipd.display(ipd.Audio(out))

## 第 10 步：下载模型（以后无限次使用）

下载这两个文件到本地（放桌面「中转站」文件夹）：
- `gtq.pth` —— 音色模型
- `added_gtq.index`（在 logs/gtq/ 下，若有 `added_*.index` 就下它，否则用 `trained_*.index`）

In [ ]:
%cd /content/rvc
from google.colab import files
import glob, shutil, os

files.download("assets/weights/gtq.pth")
idx = glob.glob("logs/gtq/added_*.index") or glob.glob("logs/gtq/trained_*.index")
if idx:
    files.download(idx[0])
else:
    print("未找到索引文件（不影响使用，相似度略降）")